In [6]:
!pip install openai
from openai import OpenAI

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import openai
import time


def get_privacy_policy_text(url):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    
    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(3)
        body_text = driver.find_element(By.TAG_NAME, 'body').text
        return body_text
    finally:
        driver.quit()


def generate_prompt(policy_text):
    prompt = f"""
You are an expert at reading privacy policies and summarizing business identity.

Based on the following privacy policy text, extract:

1. The **name of the organization**.
2. The **category or industry** the organization belongs to.
3. A **100-word description** of the company, using details inferred from the privacy policy.

Privacy Policy Text:
\"\"\"
{policy_text[:4000]}
\"\"\"

Respond in the following format:
Organization Name: ...
Category: ...
Description: ...
"""
    return prompt


from openai import OpenAI

def run_openai_chat(prompt, api_key, model="gpt-4"):
    client = OpenAI(api_key=api_key)

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that analyzes privacy policies."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=1500,
        top_p=0.95
    )

    return response.choices[0].message.content


if __name__ == "__main__":
    OPENAI_API_KEY = "API-Key"  
    url = "https://business.vericy.ai/privacy-policy"  

    policy_text = get_privacy_policy_text(url)

    prompt = generate_prompt(policy_text)

    result = run_openai_chat(prompt, OPENAI_API_KEY)

    print(result)


Organization Name: Vericy

Category: Data Privacy Services

Description: Vericy is a company committed to data privacy, providing services such as privacy policy grading. They collect minimal data from users, including email addresses and online activity data, to improve their services. They also offer a Chrome extension that collects website URLs and user interactions. The company uses Plausible, a privacy-preserving analytics tool, for understanding service usage. Vericy does not use customer data for marketing or advertising, nor do they sell it. They pride themselves on giving users control over their data, using industry-standard security measures and retaining data only as necessary.
